In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import (
    PretrainedConfig, 
    PreTrainedTokenizerFast, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import f1_score, precision_score, recall_score

import zipfile
import json
import torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


In [ ]:
# ==========================================
# 1. PHASE 3 CONFIGURATION & LABELS
# ==========================================
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
MODERN_DECODER_REPO = "A-240242/legal-modern-decoder-checkpoints" 
OUTPUT_DIR = "./phase3_modern_decoder_lorma"
HF_TOKENIZER_REPO= "A-240242/legal-modern-decoder-checkpoints"
# Official Phase 3 Labels
LABEL_LIST = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", 
    "Dissent", "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled", 
    "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"
]
NUM_LABELS = len(LABEL_LIST)
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}


In [ ]:
# ==========================================
# 2. DATA LOADING & PREPROCESSING
# ==========================================
def load_and_flatten_rr_data(split_dir, files):
    """Flattens list-based document JSONs into sentence-level (text, label) pairs."""
    sentences = []
    labels = []
    
    for filename in files:
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO, 
            filename=f"Phase_3_RR/{split_dir}/{filename}", 
            repo_type="dataset", 
            token=HF_TOKEN
        )
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        # Iterate over the list of document dicts directly
        for doc in data:
            doc_texts = doc["text"]
            doc_labels = doc["labels"]
            for text, label in zip(doc_texts, doc_labels):
                sentences.append(text)
                labels.append(LABEL2ID[label])
                
    return sentences, labels

print("Downloading and processing Phase 3 Data...")
train_texts, train_labels = load_and_flatten_rr_data("Train", ["CL_train.json", "IT_train.json"])
dev_texts, dev_labels = load_and_flatten_rr_data("Train", ["CL_dev.json", "IT_dev.json"])
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=os.environ["HF_TOKEN"])
# CRITICAL: For Causal sequence classification, right padding is standard, 
# but we must extract the hidden state of the *actual last token* before padding.
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_data(texts, labels):
    tokenized = tokenizer(texts, padding=False, truncation=True, max_length=512)
    tokenized["labels"] = labels
    return Dataset.from_dict(tokenized)

train_dataset = tokenize_data(train_texts, train_labels)
eval_dataset = tokenize_data(dev_texts, dev_labels)

In [28]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from huggingface_hub import hf_hub_download
from transformers import PretrainedConfig, Trainer, TrainingArguments, DataCollatorWithPadding
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import f1_score, precision_score, recall_score

# ==========================================
# 1. HELPER CLASSES (RoPE, RMSNorm, SwiGLU)
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(norm_x + self.eps)
        return self.weight * x_normed

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float().to(device) / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_seq_len_cached = max_position_embeddings
        t = torch.arange(self.max_seq_len_cached, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len=None):
        if seq_len > self.max_seq_len_cached:
            t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
            freqs = torch.einsum("i,j->ij", t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :])
        return (
            self.cos_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
            self.sin_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
        )

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class SwiGLUMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class LormaAdapter(nn.Module):
    def __init__(self, hidden_size, rank=128, alpha=128):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(hidden_size, rank, bias=False)
        self.B = nn.Linear(rank, hidden_size, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
    def forward(self, x):
        return self.scaling * self.B(self.A(x))

# ==========================================
# 2. BIDIRECTIONAL MODERN DECODER ARCHITECTURE
# ==========================================
class ModernAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.hidden_size // self.num_heads
        
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
        self.rotary_emb = RotaryEmbedding(self.head_dim, config.max_position_embeddings)

    def forward(self, hidden_states, attention_mask=None):
        bsz, q_len, _ = hidden_states.size()
        
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        cos, sin = self.rotary_emb(v, seq_len=q_len)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        # Build bidirectional boolean mask for SDPA if attention_mask (padding) is provided
        attn_mask = None
        if attention_mask is not None:
            # attention_mask is [bsz, seq_len], where 1 is valid, 0 is pad
            attn_mask = attention_mask.unsqueeze(1).unsqueeze(2) # [bsz, 1, 1, seq_len]
            attn_mask = attn_mask.expand(bsz, self.num_heads, q_len, q_len).bool()

        # CRITICAL: is_causal=False to allow full bidirectional context
        attn_output = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=False)
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, self.hidden_size)
        return self.o_proj(attn_output)

class ModernDecoderLormaLayer(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.self_attn = ModernAttention(config)
        self.lorma_attn = LormaAdapter(config.hidden_size, rank=rank)
        self.mlp = SwiGLUMLP(config)
        self.lorma_mlp = LormaAdapter(config.hidden_size, rank=rank)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, attention_mask=None):
        attn_out = self.self_attn(self.input_layernorm(hidden_states), attention_mask)
        hidden_states = hidden_states + attn_out + self.lorma_attn(attn_out)
        
        mlp_out = self.mlp(self.post_attention_layernorm(hidden_states))
        hidden_states = hidden_states + mlp_out + self.lorma_mlp(mlp_out)
        return hidden_states

class ModernDecoderLormaForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels, rank=128):
        super().__init__()
        self.config = config
        self.num_labels = num_labels
        
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.layers = nn.ModuleList([ModernDecoderLormaLayer(config, rank=rank) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.score = nn.Linear(config.hidden_size, num_labels, bias=False)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        hidden_states = self.embed_tokens(input_ids)
        
        for layer in self.layers:
            hidden_states = layer(hidden_states, attention_mask)
            
        hidden_states = self.norm(hidden_states)
        
        # MEAN POOLING: Average all non-padding tokens for the final sentence representation
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_embeddings = torch.sum(hidden_states * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            pooled_logits = sum_embeddings / sum_mask
        else:
            pooled_logits = hidden_states.mean(dim=1)

        logits = self.score(pooled_logits)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)


In [29]:
# ==========================================
# 3. INITIALIZATION & WEIGHT TRANSFER
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Update this repo ID to match your Phase 1 Modern Decoder repository


print("Fetching Phase 1 Pretrained Modern Decoder Checkpoint...")
config_path = hf_hub_download(repo_id=MODERN_DECODER_REPO, filename="config.json", token=HF_TOKEN)
md_config = PretrainedConfig.from_json_file(config_path)
md_config.pad_token_id = tokenizer.pad_token_id

print("Initializing Bidirectional Modern Decoder with LORMA...")
md_model = ModernDecoderLormaForSequenceClassification(md_config, num_labels=NUM_LABELS, rank=128)

model_weight_path = hf_hub_download(repo_id=MODERN_DECODER_REPO, filename="model.safetensors", token=HF_TOKEN)
md_model.load_state_dict(load_file(model_weight_path), strict=False)

# Freeze backbone, unfreeze LORMA and classification head
for param in md_model.parameters(): 
    param.requires_grad = False

for layer in md_model.layers:
    for param in layer.lorma_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_mlp.parameters(): param.requires_grad = True

for param in md_model.score.parameters(): 
    param.requires_grad = True

md_model.to(device)

trainable_params = sum(p.numel() for p in md_model.parameters() if p.requires_grad)
print(f"Trainable LORMA Parameters: {trainable_params:,}")



Fetching Phase 1 Pretrained Modern Decoder Checkpoint...


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

Initializing Bidirectional Modern Decoder with LORMA...


model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Trainable LORMA Parameters: 4,728,576


In [30]:
# ==========================================
# 5. METRICS & TRAINING EXECUTION
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Primary metric for Phase 3 is Macro-F1[cite: 11]
    macro_f1 = f1_score(labels, predictions, average="macro")
    macro_p = precision_score(labels, predictions, average="macro", zero_division=0)
    macro_r = recall_score(labels, predictions, average="macro", zero_division=0)
    return {"macro_f1": macro_f1, "macro_precision": macro_p, "macro_recall": macro_r}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ==========================================
# 4. TRAINING EXECUTION
# ==========================================
training_args_md = TrainingArguments(
    output_dir="./phase3_modern_decoder_lorma",
    report_to="none",
    num_train_epochs=10, 
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4, 
    learning_rate=1e-3, 
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="macro_f1",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    seed=42
)

md_trainer = Trainer(
    model=md_model,
    args=training_args_md,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\nCommencing Bidirectional Modern Decoder LORMA Training...")
md_trainer.train()


Commencing Bidirectional Modern Decoder LORMA Training...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Macro F1,Macro Precision,Macro Recall
1,11.851332,3.453370,0.294974,0.381441,0.284433
2,9.939733,3.590800,0.317747,0.377970,0.315949
3,8.579764,3.684444,0.335059,0.377371,0.331034
4,6.651340,4.009545,0.339965,0.376353,0.325955
5,4.698481,4.550677,0.324996,0.369433,0.315427
6,2.924328,6.073149,0.324232,0.360541,0.320503
7,1.920870,6.174601,0.333235,0.375745,0.315432
8,0.987976,6.733435,0.331895,0.361601,0.324129
9,0.724319,7.474729,0.331566,0.361658,0.325578
10,0.365182,7.770963,0.343039,0.384840,0.326314


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=2640, training_loss=4.803275343504819, metrics={'train_runtime': 2545.7096, 'train_samples_per_second': 66.17, 'train_steps_per_second': 1.037, 'total_flos': 0.0, 'train_loss': 4.803275343504819, 'epoch': 10.0})

In [31]:
# ==========================================
# 6. VALIDATION SET EVALUATION
# ==========================================
print("Running evaluation on the Development Set...")
eval_results = md_trainer.evaluate()
print(f"Validation Macro-F1: {eval_results['eval_macro_f1']:.4f}")
print(f"Validation Precision: {eval_results['eval_macro_precision']:.4f}")
print(f"Validation Recall: {eval_results['eval_macro_recall']:.4f}")

Running evaluation on the Development Set...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Macro F1,Macro Precision,Macro Recall
0.365182,7.770963,10,0.343039,0.384840,0.326314


Validation Macro-F1: 0.3430
Validation Precision: 0.3848
Validation Recall: 0.3263


In [32]:
# ==========================================
# 7. LOAD TEST DATA
# ==========================================
def load_test_data(files):
    test_docs = []
    for filename in files:
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO, 
            filename=f"Phase_3_RR/Test/{filename}", 
            repo_type="dataset", 
            token=HF_TOKEN
        )
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            if isinstance(data, dict):
                for doc_id, content in data.items():
                    # If the content is just a list of sentences
                    if isinstance(content, list):
                        test_docs.append({"id": doc_id, "text": content})
                    # If it happens to be a dictionary
                    elif isinstance(content, dict):
                        if "id" not in content:
                            content["id"] = doc_id
                        test_docs.append(content)
                        
            elif isinstance(data, list):
                test_docs.extend(data)
                
    return test_docs

print("\nDownloading Test Data...")
test_docs = load_test_data(["CL_test.json", "IT_test.json"])
print(f"Successfully loaded {len(test_docs)} test documents.")


Successfully loaded 10 test documents.


In [33]:
# ==========================================
# 8. TEST SET INFERENCE & SUBMISSION
# ==========================================
md_model .eval()
md_model .to(device)
predictions = {}

print("Generating Rhetorical Role Predictions...")
for doc in tqdm(test_docs, desc="Predicting Document Sentences"):
    doc_id = doc["id"]
    sentences = doc["text"]
    doc_predictions = []
    
    batch_size = 16
    for i in range(0, len(sentences), batch_size):
        batch_sentences = sentences[i : i + batch_size]
        
        inputs = tokenizer(
            batch_sentences, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        ).to(device)
        
        with torch.inference_mode():
            outputs = md_model(**inputs)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            
        doc_predictions.extend([ID2LABEL[pred] for pred in preds])
        
    assert len(doc_predictions) == len(sentences), f"Count mismatch for {doc_id}"
    predictions[doc_id] = doc_predictions

Generating Rhetorical Role Predictions...


Predicting Document Sentences:   0%|          | 0/10 [00:00<?, ?it/s]

In [34]:
# ==========================================
# 9. SAVE & ZIP FOR CODABENCH
# ==========================================
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"\nPhase 3 Predictions complete! Submission saved to: {SUBMISSION_ZIP}")


Phase 3 Predictions complete! Submission saved to: ./phase3_modern_decoder_lorma/submission.zip
